# REE / ESIOS demand data

Downloads and processes electricity demand data for the Spanish peninsular system.

- ESIOS 1293: actual demand
- ESIOS 1775: REE official day-ahead demand forecast

Raw observations are stored locally as JSON and converted to hourly UTC series
for use in the forecasting pipeline.

In [2]:
from pathlib import Path 
import pandas as pd
import json

from energy_forecasting.esios import (
    download_indicator,
    ACTUAL_DEMAND_INDICATOR,
    REE_D1_FORECAST_INDICATOR,
    to_hourly_series,
)

In [3]:
actual_demand_raw = download_indicator(
    indicator_id=ACTUAL_DEMAND_INDICATOR,
    start_date="2016-01-01",
    end_date="2026-09-16",
)

ree_forecast_raw = download_indicator(
    indicator_id=REE_D1_FORECAST_INDICATOR,
    start_date="2019-01-01",
    end_date="2026-09-18",
)

In [4]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

with (raw_dir / "esios_1293_actual_demand.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(actual_demand_raw, file, ensure_ascii=False)

with (raw_dir / "esios_1775_d1_forecast.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(ree_forecast_raw, file, ensure_ascii=False)

In [9]:
actual_raw_df = pd.DataFrame(actual_demand_raw)
actual_raw_df["datetime_utc"] = pd.to_datetime(
    actual_raw_df["datetime_utc"]
)

print("Actual demand raw data")
print(
    "Duplicated timestamps:",
    actual_raw_df["datetime_utc"].duplicated().sum(),
)
print(
    "Missing values:",
    actual_raw_df["value"].isna().sum(),
)

actual_raw_df["datetime_utc"].sort_values().diff().value_counts().head()

Actual demand raw data
Duplicated timestamps: 0
Missing values: 0


datetime_utc
0 days 00:05:00    453923
0 days 00:10:00    336216
Name: count, dtype: int64

In [5]:
hourly_real = to_hourly_series(actual_demand_raw)
hourly_forecast = to_hourly_series(ree_forecast_raw)

Dates passed to the ESIOS API refer to local Spanish timestamps. Processed
series use `datetime_utc` as their index, so the first UTC timestamp may fall
on the previous calendar day.

In [8]:
print("Actual demand")
print("Duplicated timestamps:", hourly_real.index.duplicated().sum())
print("Missing hours:", hourly_real.isna().sum())
print("Start:", hourly_real.index.min())
print("End:", hourly_real.index.max())

print()

print("REE D+1 forecast")
print("Duplicated timestamps:", hourly_forecast.index.duplicated().sum())
print("Missing hours:", hourly_forecast.isna().sum())
print("Start:", hourly_forecast.index.min())
print("End:", hourly_forecast.index.max())

Actual demand
Duplicated timestamps: 0
Missing hours: 0
Start: 2015-12-31 23:00:00+00:00
End: 2026-09-15 21:00:00+00:00

REE D+1 forecast
Duplicated timestamps: 0
Missing hours: 0
Start: 2018-12-31 23:00:00+00:00
End: 2026-09-17 21:00:00+00:00


In [6]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

hourly_real.to_csv(
    processed_dir / "real_demand_hourly.csv",
    index=True,
)

hourly_forecast.to_csv(
    processed_dir / "forecast_demand_hourly.csv",
    index=True,
)